# Tutorial 1: supervised fine-tuning (SFT)

In this notebook you will teach a small language model to answer maths word
problems in a fixed format: first it works through the problem inside
`<think> ... </think>`, then it gives its answer as `\boxed{...}`.

You will:

1. download a small base model, Qwen2.5-0.5B;
2. turn the GSM8K dataset of school maths problems into training conversations;
3. see exactly which tokens the model learns from;
4. measure how the model answers before training;
5. train it with one command;
6. plot the loss and measure the model again.

**What is SFT?** A language model is trained to predict the next token of a
text. Supervised fine-tuning continues that training on example
conversations that show the behaviour you want. The model is never told
whether its own answers are right; it learns to imitate the examples. This is
how a *base* model, which can only continue text, becomes one that answers in
a particular way. The repository's main SFT recipe does the same thing at a
larger scale, on 94,000 reasoning traces written by DeepSeek-R1. Training a
small model on a larger model's outputs like that is called *distillation*.

**What you need.** A TPU VM set up with `./scripts/setup_tpu_vm.sh` (see the
README), with this notebook opened on it as `examples/README.md` describes.
One TPU v6e chip is enough. Nothing else may be using the TPU while the
notebook runs.

## 1. Set up

The notebook never imports JAX itself. Training and sampling run as separate
commands through `run`, which shows their output here, so the TPU is free
again as soon as each command ends.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter starts in examples/; every command below runs from the repository
# root, as the README's do.
if not Path("pyproject.toml").is_file():
    os.chdir("..")
assert Path("src/open_r1_tpu").is_dir(), f"Not an open-r1-tpu checkout: {Path.cwd()}"
sys.path.insert(0, str(Path("examples").resolve()))
print("Working in", Path.cwd())

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from notebook_utils import count_tpu_chips, plot_scalars, read_scalars, run

PYTHON = sys.executable  # the Python that runs this notebook, and the commands

Check that JAX can see the TPU. Every chip will be used, so the number of
chips sets the recipe's *mesh*: how the model and the batch are split across
chips.

In [ ]:
CHIPS = count_tpu_chips()
MESH = f"model.mesh.shape=[{CHIPS},1]"
print(f"{CHIPS} chip(s); recipe override: {MESH}")

## 2. Download the base model

[Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) is a *base* model:
it was trained to continue text from the internet, books and code, but never
taught to follow instructions. With 0.5 billion parameters (about 1 GB) it is
small enough to train in full on one chip.

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL = Path("models/Qwen2.5-0.5B")
# The same as: hf download Qwen/Qwen2.5-0.5B --local-dir models/Qwen2.5-0.5B
snapshot_download("Qwen/Qwen2.5-0.5B", local_dir=BASE_MODEL)
sorted(path.name for path in BASE_MODEL.iterdir() if not path.name.startswith("."))

`model.safetensors` holds the weights. The `tokenizer*` files define how text
is split into tokens, and `config.json` describes the network's shape.

## 3. Build the training data

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) has 7,473 training and
1,319 test problems, each with a worked solution written by a person. Here is
the first one:

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
example = gsm8k["train"][0]
print(example["question"])
print()
print(example["answer"])

The solution has calculator notes in `<<...>>` and ends with the answer after
`####`. We rewrite it into the format we want the model to learn:

```
<think>
Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.
</think>
The answer is \boxed{72}.
```

Training data for this repository is a list of chat **messages**, each with a
`role` and some `content`. The last message must be the assistant's: it is
what the model learns to write. We also keep the bare question and answer,
which notebook 2 uses.

In [ ]:
def to_conversation(row):
    """Rewrite one GSM8K row as a user question and an assistant reply."""
    working, answer = row["answer"].split("####")
    working = re.sub(r"<<.*?>>", "", working).strip()  # drop the calculator notes
    answer = answer.strip().replace(",", "")  # "1,000" -> "1000"
    reply = f"<think>\n{working}\n</think>\nThe answer is \\boxed{{{answer}}}."
    return {
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": reply},
        ],
        "question": row["question"],
        "answer": answer,
    }


converted = gsm8k.map(to_conversation, remove_columns=["question", "answer"])
print(converted["train"][0]["messages"][1]["content"])

Save both splits as Parquet files, where the recipe expects them:

In [ ]:
DATA = Path("data/gsm8k-think")
DATA.mkdir(parents=True, exist_ok=True)
for split, rows in converted.items():
    rows.to_parquet(DATA / f"{split}.parquet")
converted

## 4. From conversation to tokens

The model never sees messages, only one long sequence of tokens. The
tokeniser's *chat template* turns a conversation into that sequence, marking
where each turn starts and ends. A *system prompt* goes first; it is the same
file the recipe uses.

In [ ]:
from transformers import AutoTokenizer

from open_r1_tpu.core.config import read_prompt_file

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
SYSTEM_PROMPT = read_prompt_file("examples/recipes/system_prompt.txt")
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    *converted["train"][0]["messages"],
]
print(tokenizer.apply_chat_template(messages, tokenize=False))

`<|im_start|>` and `<|im_end|>` are *special tokens*, a single token each.
The `<|im_end|>` that closes the assistant's turn matters most: writing it is
how the model says it has finished, and generation stops there. A model that
never learns it keeps writing until it runs out of room. `<think>`, on the
other hand, is ordinary text to this tokeniser:

In [ ]:
print(tokenizer.tokenize("<|im_end|>"))
print(tokenizer.tokenize("<think>"))

## 5. What the model learns from: the loss mask

During training the model predicts every next token, but it is only scored
(*the loss*) on the tokens of the assistant's reply. The system prompt and the
question are context: we want the model to answer questions, not to write
them. This is called *assistant-only loss*, and it is the default
(`dataset.assistant_only_loss` in the recipe).

`encode_reasoning_example` is the function training uses to turn a row into
tokens and a mask. Call it here and colour the tokens that carry loss:

In [ ]:
from itertools import groupby

from open_r1_tpu.sft.data import encode_reasoning_example

encoded = encode_reasoning_example(
    converted["train"][0], tokenizer, max_length=1024, system_prompt=SYSTEM_PROMPT
)
length = encoded.unpadded_length
tokens = encoded.input_tokens[:length].tolist()
learn = encoded.input_mask[:length].tolist()
print(f"{length} tokens, {sum(learn)} of them carry loss\n")

GREY, GREEN, RESET = "\033[90m", "\033[32m", "\033[0m"
for learned, group in groupby(zip(tokens, learn, strict=True), key=lambda p: p[1]):
    text = tokenizer.decode([token for token, _ in group])
    print((GREEN if learned else GREY) + text + RESET, end="")

The green part, the reply and its closing `<|im_end|>`, is what the model
learns. Every example is padded to `max_length` tokens so batches have a
fixed shape, and the padding carries no loss either. If the chat template
ever rendered a conversation so that the reply's start could not be found
exactly, the example would be dropped rather than trained with a guessed
mask.

## 6. Lengths and packing

An example longer than `max_length` (1,024 tokens here) is dropped, because
cutting it would teach the model answers that stop half way. Check how long
the examples are:

In [ ]:
encoded_rows = [
    encode_reasoning_example(
        row, tokenizer, max_length=1024, system_prompt=SYSTEM_PROMPT
    )
    for row in converted["train"]
]
kept = [row for row in encoded_rows if row is not None]
lengths = pd.Series([row.unpadded_length for row in kept])
print(f"Kept {len(kept)} of {len(encoded_rows)} examples.")
print(f"Median {lengths.median():.0f} tokens, longest {lengths.max()}.")
lengths.plot.hist(bins=50, title="Training examples")
plt.xlabel("tokens");

Nothing is dropped, but most examples are about 200 tokens long, so a
1,024-token window would be mostly padding. **Packing** places several
examples in one window instead. Each keeps its own positions and cannot see
the others (every token carries the number of its example, its *segment
id*), so packing makes training faster without changing what is learned.

In [ ]:
from open_r1_tpu.sft.data import pack_encoded_examples

windows = list(
    pack_encoded_examples(kept[:200], max_length=1024, pad_id=tokenizer.pad_token_id)
)
segments = np.stack([window["segment_ids"] for window in windows])
print(f"200 examples fill {len(windows)} windows, {(segments > 0).mean():.0%} full.")

# One row per window, one colour per example; padding (segment 0) is white.
plt.figure(figsize=(10, 4))
plt.imshow(
    np.ma.masked_equal(segments, 0),
    aspect="auto",
    cmap="tab10",
    interpolation="nearest",
)
plt.xlabel("token position")
plt.ylabel("window")
plt.title("200 packed examples");

Only the last windows, which take whatever is left at the end, are partly
empty.

## 7. The recipe

A training run is described by a YAML *recipe*. This tutorial's is
`examples/recipes/sft-gsm8k.yaml`; its comments explain each choice.

In [ ]:
SFT_RECIPE = "examples/recipes/sft-gsm8k.yaml"
print(Path(SFT_RECIPE).read_text())

| Section | What it sets |
| --- | --- |
| `model` | the weights to load, number precision (`bfloat16`), memory saving (`remat_config`), and the `mesh` of chips |
| `tokenizer` | the tokeniser and chat template |
| `dataset` | the data file, the window length, packing, the held-out split and the loss mask |
| `optimizer` | AdamW, and the learning rate: a short warm-up, then a cosine decay |
| `training` | the number of steps, evaluation, checkpoints and logs |
| `export` | where the trained model is written, in Hugging Face's format |

A step updates the weights once, from a *batch* of 8 windows (about 40
examples). 400 steps go through the training data about twice; each pass is
an *epoch*.

Any value can be changed on the command line with a *dotted override*, such
as `training.max_steps=100`. The recipe loader checks every key, so a typo
fails straight away instead of after the model has loaded:

In [ ]:
from open_r1_tpu.core.config import load_config
from open_r1_tpu.sft.config import validate_sft_config

config = load_config(SFT_RECIPE, [MESH], validator=validate_sft_config)
print("The recipe is valid:", config["training"]["max_steps"], "steps.")

try:
    load_config(SFT_RECIPE, ["training.max_step=100"], validator=validate_sft_config)
except ValueError as error:
    print("A typo:", error)

## 8. Before training: how does the base model answer?

`examples/answer_questions.py` loads a model onto the TPU, asks it every
GSM8K test question with the same system prompt, and scores the replies with
the reward functions in `src/open_r1_tpu/grpo/rewards.py`. It reports two
numbers:

- **correct**: the reply's final number (its `\boxed{}` answer, or else the
  last number it wrote) equals the right answer;
- **formatted**: the reply is a closed `<think>` block followed by a
  `\boxed{}` answer.

Each question is asked once and answered *greedily* (always taking the most
likely next token), so this is a quick check, not a careful benchmark.

In [ ]:
ANSWERS = Path("artifacts/gsm8k-answers")


def answer_test_questions(model_path, name):
    """Ask a model every GSM8K test question and load its scored replies."""
    output = ANSWERS / f"{name}.jsonl"
    run(
        PYTHON,
        "examples/answer_questions.py",
        model_path,
        "--model-name",
        "qwen2.5-0.5b",
        "--questions",
        "data/gsm8k-think/test.parquet",
        "--system-prompt-file",
        "examples/recipes/system_prompt.txt",
        "--output",
        output,
    )
    return pd.read_json(output, lines=True)

In [ ]:
base_answers = answer_test_questions(BASE_MODEL, "base")

In [ ]:
print(f"Correct:   {base_answers.correct.mean():.1%}")
print(f"Formatted: {base_answers.formatted.mean():.1%}")
print()
print(base_answers.reply[0])

A base model has read a lot of maths, so it may well reach the right number,
but nobody has shown it this format. It may also not stop: it was never
taught to end a turn with `<|im_end|>`, so it can carry on into a new
question of its own.

## 9. Check the setup

The *preflight* checks, in a few seconds, what would otherwise fail after the
model had loaded: that JAX sees as many chips as the mesh needs, that the
installed libraries have what training uses, and that the tokeniser gives a
clean boundary between prompt and reply.

In [ ]:
run(PYTHON, "-m", "open_r1_tpu.sft.preflight", "--config", SFT_RECIPE, MESH)

## 10. Train

This is the same command you would type in a terminal. It loads the model
onto the TPU, then repeats the training step 400 times: read a batch, compute
the loss, work out how each weight should change to lower it (the
*gradient*), and update the weights. The first step is slow because XLA
compiles the computation for the TPU; the rest are much faster. The progress
bar shows the loss, and every 50 steps the loss on the held-out examples
(`eval/loss`) is computed too. At the end the model is exported to
`artifacts/Qwen2.5-0.5B-GSM8K-SFT/merged`.

To stop early, interrupt the cell (the ■ button). Running it again carries on
from the newest checkpoint; delete `artifacts/Qwen2.5-0.5B-GSM8K-SFT` to
start again from scratch.

In [ ]:
run(PYTHON, "-m", "open_r1_tpu.sft.run", "--config", SFT_RECIPE, MESH)

## 11. The loss curve

Training logged its metrics for TensorBoard; `read_scalars` reads them into a
table.

In [ ]:
SFT_DIR = Path("artifacts/Qwen2.5-0.5B-GSM8K-SFT")
scalars = read_scalars(SFT_DIR / "logs")
plot_scalars(scalars, ["train/loss", "eval/loss"], title="SFT loss");

The loss is the average of how surprised the model is by each reply token
(its negative log-probability), so lower is better. It falls quickly at
first, while the model learns the parts every reply shares (`<think>`, the
`\boxed{}` sentence, `<|im_end|>`), then slowly, as it learns the content.
`eval/loss` comes from examples the model never trains on. If it started
rising while the training loss kept falling, the model would be memorising
its examples rather than learning from them: *overfitting*.

## 12. After training

In [ ]:
SFT_MODEL = SFT_DIR / "merged"
sft_answers = answer_test_questions(SFT_MODEL, "sft")

In [ ]:
pd.DataFrame(
    {
        "base": [base_answers.correct.mean(), base_answers.formatted.mean()],
        "after SFT": [sft_answers.correct.mean(), sft_answers.formatted.mean()],
    },
    index=["correct", "formatted"],
).style.format("{:.1%}")

Compare the two models on the same question:

In [ ]:
i = 0
print(sft_answers.question[i])
print("\nRight answer:", sft_answers.gold[i])
print("\n--- base model ---")
print(base_answers.reply[i])
print("\n--- after SFT ---")
print(sft_answers.reply[i])

The format should now be almost always right: imitation is very good at
teaching a format. Accuracy changes less, because 7,000 worked solutions are
not much maths, and the model was never told which of its own answers were
wrong. Notebook 2 trains on exactly that signal.

## Next steps

- **Continue with `02_grpo.ipynb`**, which starts from this model.
- **Talk to the model** in a terminal on the VM:

  ```bash
  python scripts/chat_tpu.py --model-path artifacts/Qwen2.5-0.5B-GSM8K-SFT/merged \
    --model-name qwen2.5-0.5b --system-prompt "$(cat examples/recipes/system_prompt.txt)"
  ```

- **Change the recipe** with overrides such as `optimizer.learning_rate=1e-5`.
  Delete `artifacts/Qwen2.5-0.5B-GSM8K-SFT` first, or the run carries on from
  its last checkpoint instead of starting again.
- **Use your own data**: any Parquet or JSON file with a `messages` column.
  `dataset.message_schema` reads other layouts, such as ShareGPT's, and
  `dataset.require_reasoning_tags=false` accepts replies without `<think>`.
- **Go bigger**: the README's quick start distils DeepSeek-R1's reasoning into
  Qwen2.5-Math-1.5B on four chips, and `docs/training.md` covers checkpoints,
  monitoring and export in detail.